# Bending playground

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.patches import Rectangle

import astropy.units as u
from astropy.time import Time
from astropy.coordinates import SkyCoord, Angle, AltAz, ICRS, EarthLocation
from astropy.coordinates import BaseCoordinateFrame, BaseRepresentation, CartesianRepresentation, SphericalRepresentation, UnitSphericalRepresentation

from astropy.coordinates import RepresentationMapping
import astropy.coordinates.representation as r
from astropy.coordinates import TimeAttribute, QuantityAttribute, EarthLocationAttribute
from erfa import ufunc as erfa_ufunc
from astropy.utils import classproperty
from astropy.coordinates.matrix_utilities import rotation_matrix
from astropy.coordinates.baseframe import frame_transform_graph

from astropy.coordinates.transformations import FunctionTransform, StaticMatrixTransform

from astropy.modeling.rotations import SphericalRotationSequence

from scipy.interpolate import CloughTocher2DInterpolator, RBFInterpolator, SmoothBivariateSpline
import scipy
scipy.__version__

import dill as pickle

from datetime import datetime
from datetime import timezone

In [3]:
df = pd.read_csv("merged.csv")

az_deg = df["Az"].values
el_deg = df["El"].values

az = np.deg2rad(az_deg)
el = np.deg2rad(el_deg)

offset_az = df["Offset Az"].values
offset_el = df["Offset El"].values

N = len(df)


# ==========================================================
# Build interleaved design matrix
# ==========================================================

X = np.zeros((2*N, 18))

for i in range(N):

    X[2*i] = [
        (1) / np.sin(el[i]), #coef 0
        (np.cos(el[i])) / np.sin(el[i]), #coef 1
        (-np.sin(el[i]) * np.sin(az[i])) / np.sin(el[i]), #coef 2
        (np.sin(el[i]) * np.cos(az[i])) / np.sin(el[i]), #coef 3
        (np.cos(el[i])**2) / np.sin(el[i]), #coef 4
        (np.sin(el[i])**2 * np.cos(2 * az[i])) / np.sin(el[i]), #coef 5
        (np.sin(el[i]) * np.cos(el[i]) * np.cos(az[i])) / np.sin(el[i]), #coef 6
        (-np.sin(el[i]) * np.cos(el[i]) * np.sin(az[i])) / np.sin(el[i]), #coef 7
        (-np.sin(el[i])**2 * np.sin(2* az[i])) / np.sin(el[i]), #coef 8
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0
    ]

    X[2*i+1] = [
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        1, #coef 9
        np.cos(el[i]),  #coef 10
        -np.sin(el[i]) * np.sin(az[i]), #coef 11
        np.sin(el[i]) * np.cos(az[i]), #coef 12
        np.cos(el[i])**2, #coef 13
        np.sin(el[i])**2 * np.cos(2 * az[i]), #coef 14
        np.sin(el[i]) * np.cos(el[i]) * np.cos(az[i]), #coef 15
        -np.sin(el[i]) * np.cos(el[i]) * np.sin(az[i]), #coef 16
        -np.sin(el[i])**2 * np.sin(2* az[i]) #coef 17 (18 total)
    ]


print("Design Matrix Shape:", X.shape)
print(X)


# ==========================================================
# Build matching target vector
# ==========================================================

y = np.zeros(2*N)

for i in range(N):
    y[2*i] = offset_az[i]
    y[2*i+1] = offset_el[i]

print(y)


# ==========================================================
# Solve least squares
# ==========================================================

coef, residuals, rank, s = np.linalg.lstsq(X, y, rcond=None)

print("\nCoefficients:")
print(coef)

# ==========================================================
# Prediction functions
# ==========================================================

def azfit_trig(az_deg, el_deg=None, grid=False):

    az = np.deg2rad(np.asarray(az_deg))

    return (
        coef[0] / np.sin(el[i])
        + coef[1] * ((np.cos(el[i])) / np.sin(el[i]))
        + coef[2] * ((-np.sin(el[i]) * np.sin(az[i])) / np.sin(el[i]))
        + coef[3] * ((np.sin(el[i]) * np.cos(az[i])) / np.sin(el[i]))
        + coef[4] * ((np.cos(el[i])**2) / np.sin(el[i]))
        + coef[5] * ((np.sin(el[i])**2 * np.cos(2 * az[i])) / np.sin(el[i]))
        + coef[6] * ((np.sin(el[i]) * np.cos(el[i]) * np.cos(az[i])) / np.sin(el[i]))
        + coef[7] * ((-np.sin(el[i]) * np.cos(el[i]) * np.sin(az[i])) / np.sin(el[i]))
        + coef[8] * ((-np.sin(el[i])**2 * np.sin(2* az[i])) / np.sin(el[i]))
    )


def altfit_trig(az_deg=None, el_deg=None, grid=False):

    el = np.deg2rad(np.asarray(el_deg))

    return (
        + coef[9]
        + coef[10] * (np.cos(el[i]))
        + coef[11] * (-np.sin(el[i]) * np.sin(az[i]))
        + coef[12] * (np.sin(el[i]) * np.cos(az[i]))
        + coef[13] * (np.cos(el[i])**2)
        + coef[14] * (np.sin(el[i])**2 * np.cos(2 * az[i]))
        + coef[15] * (np.sin(el[i]) * np.cos(el[i]) * np.cos(az[i]))
        + coef[16] * (-np.sin(el[i]) * np.cos(el[i]) * np.sin(az[i]))
        + coef[17] * (-np.sin(el[i])**2 * np.sin(2* az[i]))
    )

Design Matrix Shape: (206, 18)
[[ 1.08799294  0.42863579 -0.91448216 ...  0.          0.
   0.        ]
 [ 0.          0.          0.         ...  0.14651777 -0.3311399
  -0.62518291]
 [ 1.08429018  0.41914818 -0.91326339 ...  0.          0.
   0.        ]
 ...
 [ 0.          0.          0.         ...  0.48163316  0.13411641
   0.2617844 ]
 [ 1.49381656  1.10972425  0.26290325 ...  0.          0.
   0.        ]
 [ 0.          0.          0.         ...  0.47980838  0.13074241
   0.22734149]]
[ 0.067 -0.191  0.068 -0.192  0.066 -0.19   0.069 -0.189  0.071 -0.186
  0.073 -0.184  0.073 -0.183  0.072 -0.183  0.077 -0.179  0.075 -0.186
  0.075 -0.189  0.072 -0.193  0.07  -0.201  0.056 -0.221  0.042 -0.23
  0.023 -0.234  0.004 -0.24  -0.019 -0.248 -0.035 -0.251 -0.044 -0.246
 -0.059 -0.245 -0.067 -0.247 -0.073 -0.248 -0.077 -0.246 -0.079 -0.248
 -0.083 -0.248 -0.084 -0.249 -0.087 -0.25  -0.088 -0.25  -0.09  -0.254
 -0.091 -0.253 -0.092 -0.257 -0.094 -0.259 -0.096 -0.263 -0.097 -0.268
 -0.1 

In [4]:
class bending_model:
    def __init__(self, altfit_func, azfit_func, model_name, model_description):
        self.alt_func = altfit_func
        self.az_func = azfit_func
        self.name = model_name
        self.description = model_description

    def astro_to_drive(self, astro_alt, astro_az):
        deg_alt = Angle(astro_alt*u.deg).wrap_at(180*u.deg).deg
        deg_az = Angle(astro_az*u.deg).wrap_at(360*u.deg).deg
        drive_alt = (astro_alt-self.alt_func(deg_az, deg_alt, grid=False))
        drive_az = (astro_az-self.az_func(deg_az, deg_alt, grid=False))
        return (drive_alt, drive_az)

    def drive_to_astro(self, drive_alt, drive_az):
        deg_alt = Angle(drive_alt*u.deg).wrap_at(180*u.deg).deg
        deg_az = Angle(drive_az*u.deg).wrap_at(360*u.deg).deg
        astro_alt = (drive_alt+self.alt_func(deg_az, deg_alt, grid=False))
        astro_az = (drive_az+self.az_func(deg_az, deg_alt, grid=False))
        return (astro_alt, astro_az)

    def delta(self, alt, az):
        deg_alt = Angle(alt*u.deg).wrap_at(180*u.deg).deg
        deg_az = Angle(az*u.deg).wrap_at(360*u.deg).deg
        delta_alt = self.alt_func(deg_az, deg_alt, grid=False)
        delta_az = self.az_func(deg_az, deg_alt, grid=False)
        return (delta_alt, delta_az)

In [5]:
b = bending_model(
    altfit_func=altfit_trig,
    azfit_func=azfit_trig,
    model_name='testModel',
    model_description='blank'
)

"""
b = bending_model(altfit_func=altfit_spline,
                  azfit_func=azfit_spline,
                  model_name='test_model_v0',
                  model_description=f'generated on {datetime.now().strftime("%Y/%m/%d %H:%M:%S")} using toy data and scipy {scipy.__version__} SmoothBivariateSpline'
                 )
"""

'\nb = bending_model(altfit_func=altfit_spline,\n                  azfit_func=azfit_spline,\n                  model_name=\'test_model_v0\',\n                  model_description=f\'generated on {datetime.now().strftime("%Y/%m/%d %H:%M:%S")} using toy data and scipy {scipy.__version__} SmoothBivariateSpline\'\n                 )\n'

In [6]:
##Chi squared test

#pred_alt = altfit_trig(df["El"].values)
#pred_az  = azfit_trig(df["Az"].values)
az = np.deg2rad(df["Az"].values)
el = np.deg2rad(df["El"].values)

az_pred = (
coef[0] / np.sin(el)
+ coef[1] * ((np.cos(el)) / np.sin(el))
+ coef[2] * ((-np.sin(el) * np.sin(az)) / np.sin(el))
+ coef[3] * ((np.sin(el) * np.cos(az)) / np.sin(el))
+ coef[4] * ((np.cos(el)**2) / np.sin(el))
+ coef[5] * ((np.sin(el)**2 * np.cos(2 * az)) / np.sin(el))
+ coef[6] * ((np.sin(el) * np.cos(el) * np.cos(az)) / np.sin(el))
+ coef[7] * ((-np.sin(el) * np.cos(el) * np.sin(az)) / np.sin(el))
+ coef[8] * ((-np.sin(el)**2 * np.sin(2* az)) / np.sin(el))
)


el_pred = (
coef[9]
+ coef[10] * np.cos(el)
+ coef[11] * (-np.sin(el) * np.sin(az))
+ coef[12] * (np.sin(el) * np.cos(az))
+ coef[13] * (np.cos(el)**2)
+ coef[14] * (np.sin(el)**2 * np.cos(2 * az))
+ coef[15] * (np.sin(el) * np.cos(el) * np.cos(az))
+ coef[16] * (-np.sin(el) * np.cos(el) * np.sin(az))
+ coef[17] * (-np.sin(el)**2 * np.sin(2* az))
)

df["Model Offset El"] = el_pred
#print(el_pred)
df["Model Offset Az"] = az_pred
#print(az_pred)


df["El Residual"] = df["Offset El"] - df["Model Offset El"]
df["Az Residual"] = df["Offset Az"] - df["Model Offset Az"]

chi2_el = np.sum(df["El Residual"]**2)
chi2_az = np.sum(df["Az Residual"]**2)
chi2_total = chi2_el + chi2_az

##sigma here is calculated to be variance of the residuals; later, consider variance in...
##fitted parameters
##predictions

"""
N = len(df)
p = 3  #parameters per axis
sigma_el = np.sqrt(np.sum(df["El Residual"]**2) / (N - p))
sigma_az = np.sqrt(np.sum(df["Az Residual"]**2) / (N - p))
print("Estimated sigma El:", sigma_el)
print("Estimated sigma Az:", sigma_az)
"""

print("El RMS:", np.sqrt(np.mean(df["El Residual"]**2)))
print("Az RMS:", np.sqrt(np.mean(df["Az Residual"]**2)))
print("|d| RMS:", np.sqrt(np.mean(df["Az Residual"]**2)+ df["El Residual"]**2))

print("\nIndividual Residuals:")
for index, row in df.iterrows():
    print(f"  El Residual: {row['El Residual']:.6f}, Az Residual: {row['Az Residual']:.6f}")

El RMS: 0.008490686283631746
Az RMS: 0.004209458656474698
|d| RMS: 0      0.006423
1      0.007541
2      0.006174
3      0.005994
4      0.004727
         ...   
98     0.008208
99     0.006535
100    0.005954
101    0.005201
102    0.006392
Name: El Residual, Length: 103, dtype: float64

Individual Residuals:
  El Residual: -0.004851, Az Residual: 0.002211
  El Residual: -0.006256, Az Residual: 0.002535
  El Residual: -0.004516, Az Residual: 0.000073
  El Residual: -0.004267, Az Residual: 0.001568
  El Residual: -0.002150, Az Residual: 0.001143
  El Residual: -0.000503, Az Residual: 0.001422
  El Residual: 0.000509, Az Residual: -0.001239
  El Residual: 0.000775, Az Residual: -0.003332
  El Residual: 0.005551, Az Residual: 0.000127
  El Residual: 0.001682, Az Residual: -0.003951
  El Residual: 0.002493, Az Residual: -0.003966
  El Residual: 0.003583, Az Residual: -0.004987
  El Residual: 0.003099, Az Residual: -0.001190
  El Residual: -0.005023, Az Residual: -0.000028
  El Residual: 

In [7]:
"""
##10 parameter model
pred_alt = altfit_trig(df["Az"].values, df["El"].values)
pred_az  = azfit_trig(df["Az"].values, df["El"].values)

df["Model Offset El"] = pred_alt
df["Model Offset Az"] = pred_az

df["El Residual"] = df["Offset El"] - df["Model Offset El"]
df["Az Residual"]  = df["Offset Az"] - df["Model Offset Az"]

print("El RMS:", np.sqrt(np.mean(df["El Residual"]**2)))
print("Az RMS:", np.sqrt(np.mean(df["Az Residual"]**2)))

az_test = 292.689301  # degrees
el_test = 61.201706    # degrees

d_el = altfit_trig(az_test, el_test)
d_az = azfit_trig(az_test, el_test)

print("Predicted offsets:")
print("ΔAz:", d_az)
print("ΔEl:", d_el)
"""
az = np.deg2rad(df["Az"].values)
el = np.deg2rad(df["El"].values)

# Predictions
az_pred = (
coef[0] / np.sin(el)
+ coef[1] * ((np.cos(el)) / np.sin(el))
+ coef[2] * ((-np.sin(el) * np.sin(az)) / np.sin(el))
+ coef[3] * ((np.sin(el) * np.cos(az)) / np.sin(el))
+ coef[4] * ((np.cos(el)**2) / np.sin(el))
+ coef[5] * ((np.sin(el)**2 * np.cos(2 * az)) / np.sin(el))
+ coef[6] * ((np.sin(el) * np.cos(el) * np.cos(az)) / np.sin(el))
+ coef[7] * ((-np.sin(el) * np.cos(el) * np.sin(az)) / np.sin(el))
+ coef[8] * ((-np.sin(el)**2 * np.sin(2* az)) / np.sin(el))
)


el_pred = (
coef[9]
+ coef[10] * np.cos(el)
+ coef[11] * (-np.sin(el) * np.sin(az))
+ coef[12] * (np.sin(el) * np.cos(az))
+ coef[13] * (np.cos(el)**2)
+ coef[14] * (np.sin(el)**2 * np.cos(2 * az))
+ coef[15] * (np.sin(el) * np.cos(el) * np.cos(az))
+ coef[16] * (-np.sin(el) * np.cos(el) * np.sin(az))
+ coef[17] * (-np.sin(el)**2 * np.sin(2* az))
)

for az_p, el_p in zip(az_pred, el_pred):
    print(f"PrEl Offset: {el_p:.6f}  PrAz Offset: {az_p:.6f}")

"""
##6 parameter model
az = np.deg2rad(df["Az"].values)
el = np.deg2rad(df["El"].values)

# Predictions using 3-term formulas
az_pred = coef[0] + coef[1]*np.cos(az) + coef[2]*np.sin(az)
el_pred = coef[3] + coef[4]*np.cos(el) + coef[5]*np.sin(el)

for i in range(len(df)):
    print("PrEl Offset:", el_pred[i], "PrAz Offset:", az_pred[i])
"""

"""
pred_alt = altfit_trig(df["El"].values)
pred_az  = azfit_trig(df["Az"].values)

df["Model Offset El"] = pred_alt
df["Model Offset Az"] = pred_az

df["El Residual"] = df["Offset El"] - df["Model Offset El"]
df["Az Residual"] = df["Offset Az"] - df["Model Offset Az"]

print("El RMS:", np.sqrt(np.mean(df["El Residual"]**2)))
print("Az RMS:", np.sqrt(np.mean(df["Az Residual"]**2)))

az_test = 65.960297 # degrees
el_test = 67.259094   # degrees

d_el = altfit_trig(el_test)
d_az = azfit_trig(az_test)

print("Predicted offsets:")
print("ΔAz:", d_az)
print("ΔEl:", d_el)
"""



PrEl Offset: -0.186149  PrAz Offset: 0.064789
PrEl Offset: -0.185744  PrAz Offset: 0.065465
PrEl Offset: -0.185484  PrAz Offset: 0.065927
PrEl Offset: -0.184733  PrAz Offset: 0.067432
PrEl Offset: -0.183850  PrAz Offset: 0.069857
PrEl Offset: -0.183497  PrAz Offset: 0.071578
PrEl Offset: -0.183509  PrAz Offset: 0.074239
PrEl Offset: -0.183775  PrAz Offset: 0.075332
PrEl Offset: -0.184551  PrAz Offset: 0.076873
PrEl Offset: -0.187682  PrAz Offset: 0.078951
PrEl Offset: -0.191493  PrAz Offset: 0.078966
PrEl Offset: -0.196583  PrAz Offset: 0.076987
PrEl Offset: -0.204099  PrAz Offset: 0.071190
PrEl Offset: -0.215977  PrAz Offset: 0.056028
PrEl Offset: -0.224252  PrAz Offset: 0.040614
PrEl Offset: -0.230306  PrAz Offset: 0.025621
PrEl Offset: -0.236935  PrAz Offset: 0.001784
PrEl Offset: -0.240284  PrAz Offset: -0.019862
PrEl Offset: -0.241475  PrAz Offset: -0.037091
PrEl Offset: -0.241739  PrAz Offset: -0.045204
PrEl Offset: -0.242137  PrAz Offset: -0.057667
PrEl Offset: -0.242648  PrAz O

'\npred_alt = altfit_trig(df["El"].values)\npred_az  = azfit_trig(df["Az"].values)\n\ndf["Model Offset El"] = pred_alt\ndf["Model Offset Az"] = pred_az\n\ndf["El Residual"] = df["Offset El"] - df["Model Offset El"]\ndf["Az Residual"] = df["Offset Az"] - df["Model Offset Az"]\n\nprint("El RMS:", np.sqrt(np.mean(df["El Residual"]**2)))\nprint("Az RMS:", np.sqrt(np.mean(df["Az Residual"]**2)))\n\naz_test = 65.960297 # degrees\nel_test = 67.259094   # degrees\n\nd_el = altfit_trig(el_test)\nd_az = azfit_trig(az_test)\n\nprint("Predicted offsets:")\nprint("ΔAz:", d_az)\nprint("ΔEl:", d_el)\n'

In [ ]:
#For CalculatedOffsets
df = pd.read_csv('TrainedData.csv')
plt.scatter(df['May Az'], df['May El'], color='blue', marker='o', s=5, label= 'May')
#plt.scatter(df['May Calc Az'], df['May Calc El'], color='red', marker='o', s=5, label= 'May Calc')
plt.scatter(df['June Az'], df['June El'], color='green', marker='o', s=5, label= 'June')
#plt.scatter(df['June Calc Az'], df['June Calc El'], color='orange', marker='o', s=5, label= 'June Calc')
plt.xlabel('Azimuth (degree)')
plt.ylabel('Elevation (degree)')
plt.title("Actual Offsets - 8x Scale")
plt.grid(True, linestyle='--', color='gray', linewidth=0.5)
plt.legend()


Scale = 8
plt.quiver(
    df['May Az'],
    df['May El'],
    (df['May Calc Az'] - df['May Az']) * Scale,
    (df['May Calc El'] - df['May El']) * Scale,
    color='black',
    width=0.003,
    angles='xy',
    scale_units='xy',
    scale=.75
)

plt.quiver(
    df['June Az'],
    df['June El'],
    (df['June Calc Az'] - df['June Az']) * Scale,
    (df['June Calc El'] - df['June El']) * Scale,
    color='black',
    width=0.003,
    angles='xy',
    scale_units='xy',
    scale=.75
)

plt.savefig('CalcOffsets.png')
plt.close()

In [ ]:
#For Predicted Offsets
plt.scatter(df['May Az'], df['May El'], color='blue', marker='o', s=5, label= 'May')
#plt.scatter(df['May PrAz'], df['May PrEl'], color='red', marker='o', s=5, label= 'May Bending')
#plt.scatter(df['May Calc Az'], df['May Calc El'], color='red', marker='o', s=5, label= 'May Calc')
plt.scatter(df['June Az'], df['June El'], color='green', marker='o', s=5, label= 'June')
#plt.scatter(df['June PrAz'], df['June PrEl'], color='orange', marker='o', s=5, label= 'June Bending')
#plt.scatter(df['June Calc Az'], df['June Calc El'], color='orange', marker='o', s=5, label= 'June Calc')
plt.xlabel('Azimuth (degree)')
plt.ylabel('Elevation (degree)')
plt.grid(True, linestyle='--', color='gray', linewidth=0.5)
plt.title("Predicted Offsets - 8x Scale")
plt.legend()

Scale = 8
plt.quiver(
    df['May Az'],
    df['May El'],
    (df['May PrAz'] - df['May Az']) * Scale,
    (df['May PrEl'] - df['May El']) * Scale,
    color='black',
    width=0.003,
    angles='xy',
    scale_units='xy',
    scale=.75
)

plt.quiver(
    df['June Az'],
    df['June El'],
    (df['June PrAz'] - df['June Az']) * Scale,
    (df['June PrEl'] - df['June El']) * Scale,
    color='black',
    width=0.003,
    angles='xy',
    scale_units='xy',
    scale=.75
)


plt.savefig('Predicted Offsets.png')
plt.close()